In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 48
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## QoG Pipeline

**Source:** Quality of Government Institute — Standard Time-Series Dataset
**Access:** Automated direct CSV download — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — QOG section

### Sources covered by this pipeline
| Source | QoG Variables | Concept |
|--------|--------------|---------|
| KOF Economic Globalisation | dr_eg | Trade governance (proxy — see framework_decisions.md) |
| Political Terror Scale | gd_ptsa, gd_ptsh, gd_ptss | Personal security, Civil liberties |
| Open Budget Survey | ibp_obi | PFM, Government transparency |
| ND-GAIN | gain_gov, gain_read | Environmental/climate governance |
| Bayesian Corruption Index | bci_bci | Control of corruption (cross-check) |
| Hanson-Sigman State Capacity | lld_capacity | State capacity (supplementary) |
| Comparative Constitutions Project | ccp_syst, ccp_market, ccp_civil, ccp_infoacc, ccp_equal | Legal quality, Property rights, Legislative checks |
| Electoral Integrity Project | pei_peii_1 | Electoral process |
| Global Peace Index | gpi_gpi | Political stability (optional cross-check) |
| WB Informal Economy Database | ied_mimic, ied_dge | State capacity |

In [2]:
import requests
import io
from datetime import datetime

QOG_BASE = "https://www.qogdata.pol.gu.se/data"

def get_latest_qog_version():
    """Auto-detect latest QoG version by trying recent years."""
    current_year = datetime.today().year
    for year in range(current_year, current_year - 3, -1):
        yy = str(year)[-2:]
        url = f"{QOG_BASE}/qog_std_ts_jan{yy}.csv"
        response = requests.head(url, timeout=10, allow_redirects=True)
        if response.status_code == 200 and 'text/csv' in response.headers.get('Content-Type', ''):
            print(f"Latest QoG version: Jan{yy} ({year})")
            return yy
    return None

QOG_VERSION = get_latest_qog_version()
print(f"Using version: Jan{QOG_VERSION}")

Latest QoG version: Jan26 (2026)
Using version: Jan26


In [3]:
# Download QoG Standard Time-Series dataset
# This is the main data download — ~68MB, takes 1-2 minutes
print("Downloading QoG Standard Time-Series dataset...")
url = f"{QOG_BASE}/qog_std_ts_jan{QOG_VERSION}.csv"
response = requests.get(url, timeout=300)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024/1024:.1f} MB")

qog_std = pd.read_csv(io.StringIO(response.text), low_memory=False)
print(f"\nShape: {qog_std.shape}")
print(f"Years: {qog_std['year'].min()} — {qog_std['year'].max()}")
print(f"Countries: {qog_std['cname'].nunique()}")

Status: 200, Size: 68.3 MB



Shape: (12585, 1813)
Years: 1946 — 2025
Countries: 204


In [4]:
# Check file sizes before downloading
QOG_BASE = "https://www.qogdata.pol.gu.se/data"

def get_latest_qog_version():
    """Auto-detect latest QoG version by trying recent years."""
    current_year = datetime.today().year
    for year in range(current_year, current_year - 3, -1):
        yy = str(year)[-2:]
        url = f"{QOG_BASE}/qog_std_ts_jan{yy}.csv"
        response = requests.head(url, timeout=10, allow_redirects=True)
        if response.status_code == 200 and 'text/csv' in response.headers.get('Content-Type', ''):
            size_mb = int(response.headers.get('Content-Length', 0)) / (1024*1024)
            print(f"Latest QoG version: Jan{yy} ({year})")
            return yy, year
    return None, None

QOG_VERSION, QOG_YEAR = get_latest_qog_version()

# Check sizes of both Standard and Basic datasets
for dataset in ['std', 'bas']:
    url = f"{QOG_BASE}/qog_{dataset}_ts_jan{QOG_VERSION}.csv"
    response = requests.head(url, timeout=10, allow_redirects=True)
    size_mb = int(response.headers.get('Content-Length', 0)) / (1024*1024)
    print(f"qog_{dataset}_ts_jan{QOG_VERSION}.csv: {size_mb:.1f} MB")

Latest QoG version: Jan26 (2026)


qog_std_ts_jan26.csv: 68.3 MB


qog_bas_ts_jan26.csv: 14.9 MB


In [5]:
QOG_VARIABLES = {
    'cname': 'country_name',
    'ccodecow': 'cow_code',
    'ccodealp': 'country_code',
    'year': 'year',
    'dr_eg': 'kof_economic_globalisation',
    'gd_ptsa': 'pts_amnesty',
    'gd_ptsh': 'pts_hrw',
    'gd_ptss': 'pts_statedept',
    'ibp_obi': 'obs_open_budget_index',
    'gain_gov': 'nd_gain_governance_readiness',
    'gain_read': 'nd_gain_readiness',
    'bci_bci': 'bci_corruption_index',
    'lld_capacity': 'hanson_sigman_state_capacity',
    'ccp_syst': 'ccp_government_system',
    'ccp_market': 'ccp_market_economy_provisions',
    'ccp_civil': 'ccp_civil_rights_provisions',
    'ccp_infoacc': 'ccp_information_access',
    'ccp_equal': 'ccp_equality_provisions',
    'pei_peii_1': 'pei_electoral_integrity_index',
    'gpi_gpi': 'gpi_peace_index',
    # WB Informal Economy Database — State capacity proxy (administrative reach)
    'ied_mimic': 'wb_informal_economy_mimic',
    'ied_dge':   'wb_informal_economy_dge',
    # Romelli Central Bank Independence Extended (CBIE) index — Macroeconomic policy
    'cbie_index':   'romelli_cbi_index',
    'cbie_policy':  'romelli_cbi_policy',
    'cbie_lending': 'romelli_cbi_lending',
    
    # Polity5 — regime type and durability (supplementary for Electoral process, Legislative checks)
    'p_polity2':  'polity5_score',
    'p_durable':  'polity5_regime_durability',
    
    # NELDA — National Elections Across Democracy and Autocracy (Electoral process supplementary)
    'nelda_fme':   'nelda_concerns_not_free_fair',   # NELDA11: concerns elections wont be free/fair (1=BAD)
    'nelda_mbbe':  'nelda_media_bias_incumbent',     # NELDA16: media bias favoring incumbent (1=BAD)
    'nelda_mtop':  'nelda_mtop_low_signal',          # near-zero discrimination (0.964 vs 1.000) - NOT scored
    'nelda_noe':   'nelda_election_held',
    'nelda_noea':  'nelda_executive_election',
    'nelda_noel':  'nelda_legislative_election',
    'nelda_oa':    'nelda_opposition_allowed',       # opposition allowed (1=GOOD)
    'nelda_rpae':  'nelda_riots_protests_after',     # riots/protests after election (1=BAD)
    'nelda_vcdbe': 'nelda_violence_deaths_before',   # NELDA33: violence/civilian deaths before election (1=BAD)
}

missing_cols = [col for col in QOG_VARIABLES.keys() if col not in qog_std.columns]
if missing_cols:
    print(f"⚠️ Missing columns: {missing_cols}")
else:
    print("All selected columns present ✅")
print(f"\nSelecting {len(QOG_VARIABLES) - 4} indicators plus 4 identifiers")

All selected columns present ✅

Selecting 32 indicators plus 4 identifiers


In [6]:
# Filter to selected columns and rename
qog = qog_std[list(QOG_VARIABLES.keys())].copy()
qog = qog.rename(columns=QOG_VARIABLES)

# NELDA: raw 99 = 'unclear/could not code' = MISSING, not a real value.
# Convert to NaN before any use, or aggregations would be swamped by 99s.
_nelda_cols = [c for c in qog.columns if c.startswith('nelda_') and c not in
               ('nelda_election_held','nelda_executive_election','nelda_legislative_election')]
qog[_nelda_cols] = qog[_nelda_cols].replace(99, pd.NA)

# Filter to framework start year
qog = qog[qog['year'] >= FRAMEWORK_START_YEAR].copy()

# Sort
qog = qog.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {qog.shape}")
print(f"Years: {qog['year'].min()} — {qog['year'].max()}")
print(f"Countries: {qog['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (qog.isnull().sum() / len(qog) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(qog.head(3))

Shape: (6889, 36)
Years: 1990 — 2025
Countries: 200

Missing values (%):
pei_electoral_integrity_index    92.0
obs_open_budget_index            86.8
pts_hrw                          83.8
nelda_opposition_allowed         83.6
nelda_mtop_low_signal            83.5
nelda_media_bias_incumbent       78.9
nelda_violence_deaths_before     77.5
nelda_riots_protests_after       77.1
nelda_executive_election         75.5
nelda_concerns_not_free_fair     75.5
nelda_election_held              75.5
nelda_legislative_election       75.5
gpi_peace_index                  58.2
hanson_sigman_state_capacity     39.1
wb_informal_economy_mimic        35.8
polity5_score                    32.3
polity5_regime_durability        31.5
wb_informal_economy_dge          30.5
pts_amnesty                      29.1
romelli_cbi_index                27.3
romelli_cbi_lending              27.3
romelli_cbi_policy               27.3
nd_gain_governance_readiness     21.4
nd_gain_readiness                19.8
bci_corruption_

In [7]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "qog_clean.csv")
qog.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {qog.shape}")

# Get latest year from data
latest_year = str(int(qog['year'].max()))

# Update download log
update_entry(
    "GPI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: gpi_gpi."
)

update_entry(
    "PTS",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variables: gd_ptsa (Amnesty), gd_ptsh (HRW), gd_ptss (State Dept)."
)

update_entry(
    "OBS",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: ibp_obi (Open Budget Index)."
)

update_entry(
    "ND_GAIN",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variables: gain_gov (governance readiness), gain_read (readiness). Master PDF specifies these sub-scores not overall index."
)

update_entry(
    "BCI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: bci_bci."
)

update_entry(
    "HANSON_SIGMAN",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: lld_capacity. Double-counting caveat: incorporates V-Dem and other sources we use."
)

update_entry(
    "CCP",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variables: ccp_syst, ccp_market, ccp_civil, ccp_infoacc, ccp_equal. Note: judicial independence and separation of powers sub-dimensions not clearly captured in QoG CCP variable subset — gap flagged."
)

update_entry(
    "PEI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: pei_peii_1. Per-election cadence — high missingness expected."
)

update_entry(
    "KOF_TRADE",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="qog_clean.csv",
    latest_available_version=f"QoG Jan{QOG_VERSION}",
    notes="Sourced via QoG Standard TS dataset. Variable: dr_eg (Economic Globalisation). MISMATCH: master PDF calls for Trade Globalization subindex specifically. dr_eg covers trade+financial. Decision flagged in framework_decisions.md."
)
update_entry(
    "WB_INFORMAL",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date="2020",
    local_filename="qog_clean.csv",
    latest_available_version="QoG Jan26",
    notes="Sourced via QoG Standard TS dataset. Variables: ied_mimic (MIMIC model), ied_dge (DGE model). Coverage: 1990-2020, ~180 countries. Proxy for state administrative reach."
)
print_entry("WB_INFORMAL")
update_entry(
    "ROMELLI_CBI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date="2023",
    local_filename="qog_clean.csv",
    latest_available_version="QoG Jan26",
    notes="Sourced via QoG Standard TS dataset. Variables: cbie_index (overall), cbie_policy, cbie_lending. Coverage: 1923-2023, 155 countries."
)
print_entry("ROMELLI_CBI")
update_entry(
    "POLITY5",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date="2020",
    local_filename="qog_clean.csv",
    latest_available_version="QoG Jan26",
    notes="Sourced via QoG Standard TS dataset. Variables: p_polity2 (Polity score -10 to +10), p_durable (regime durability). Coverage: 1946-2020. Note: Polity project not updated since ~2018; QoG version is as current as source."
)

update_entry(
    "NELDA",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date="2020",
    local_filename="qog_clean.csv",
    latest_available_version="QoG Jan26",
    notes="Sourced via QoG Standard TS dataset. 10 variables covering election occurrence, competitiveness, fairness, opposition access. Coverage: per-election cadence, ~2000-2020. Note: NELDA latest release is 2020; QoG version is as current as source."
)

print_entry("POLITY5")
print_entry("NELDA")

print("\nAll log entries updated.")
print_entry("KOF_TRADE")

Written: C:\Users\mjbou\governance-framework\data\processed\qog_clean.csv
Shape: (6889, 36)
[download_log] Updated entry for GPI
[download_log] Updated entry for PTS
[download_log] Updated entry for OBS
[download_log] Updated entry for ND_GAIN
[download_log] Updated entry for BCI
[download_log] Updated entry for HANSON_SIGMAN
[download_log] Updated entry for CCP
[download_log] Updated entry for PEI
[download_log] Updated entry for KOF_TRADE
[download_log] Updated entry for WB_INFORMAL


  source_id: WB_INFORMAL
  last_attempted_date: 2026-06-07
  last_successful_download_date: 2026-07-29
  data_as_of_date: 2020
  local_filename: qog_clean.csv
  latest_available_version: QoG Jan26
  no_update_reason: nan
  notes: Sourced via QoG Standard TS dataset. Variables: ied_mimic (MIMIC model), ied_dge (DGE model). Coverage: 1990-2020, ~180 countries. Proxy for state administrative reach.
[download_log] Updated entry for ROMELLI_CBI
  source_id: ROMELLI_CBI
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-07-29
  data_as_of_date: 2023
  local_filename: qog_clean.csv
  latest_available_version: QoG Jan26
  no_update_reason: nan
  notes: Sourced via QoG Standard TS dataset. Variables: cbie_index (overall), cbie_policy, cbie_lending. Coverage: 1923-2023, 155 countries.
[download_log] Updated entry for POLITY5
[download_log] Updated entry for NELDA
  source_id: POLITY5
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-07-29
  data_as_of_d